In [1]:
import numpy as np
import pandas as pd

In [2]:
def import_excel(start_year, end_year):
    df = pd.read_excel('./migr/migr_combined_2007-2019.xlsx', sheet_name=[str(i) for i in range(start_year, end_year+1)])  
    df = pd.concat(df, axis=0, ignore_index=True)
    return df

In [3]:
migr_2009_2014 = import_excel(2009, 2014)
migr_2015_2019 = import_excel(2015, 2019)

In [4]:
col_mapping = {
    'Insgesamt': 'total_foreigners',
    'Jahr': 'year',
    'Europäische Union (EU)': 'EU',
    'Polen': 'POL',
    'Italien': 'ITA',
    'ehemalige Sowjetunion': 'former_UDSSR',
    'Russische Föderation': 'RUS',
    'Ukraine': 'UKR',
    'Kasachstan': 'KAZ',
    'ehemaliges Jugoslawien': 'former_JUGO',
    'Türkei': 'TUR',
    'Arabische Länder': 'Arabic_countries',
    'Vietnam': 'VNM',
    'Vereinigte Staaten/USA': 'USA',
    'Frankreich': 'FRA',
    'Griechenland': 'GRC',
    'Österreich': 'AUT',
    'Vereinigtes Königreich': 'UK',
    'Bulgarien': 'BGR',
    'Bosnien und Herzegowina': 'BIH',
    'Kroatien': 'HRV',
    'Serbien': 'SRB',
    'Islamische Länder': 'Islamic_countries',
    'Iran': 'IRN',
    'Libanon': 'LBN',
    'nicht eindeutig zuordenbar': 'not_identified',
    'Spanien': 'ESP',
    'Rumänien': 'ROU',
    'China': 'CHN',
    'Syrien': 'SYR'
}

### 2015-2019

In [5]:
migr_2015_2019['PR'] = migr_2015_2019['Planungsraum'].astype(int).apply('{:0>8}'.format)
migr_2015_2019['LOR'] = migr_2015_2019['PR'].apply(lambda x: str(x)[:6])
migr_2015_2019['Bezirk'] = migr_2015_2019['PR'].apply(lambda x: str(x)[:2])
migr_2015_2019['Prognoseraum'] = migr_2015_2019['PR'].apply(lambda x: str(x)[2:4])
migr_2015_2019['Bezirksregion'] = migr_2015_2019['PR'].apply(lambda x: str(x)[4:6])
migr_2015_2019['Planungsraum'] = migr_2015_2019['PR'].apply(lambda x: str(x)[6:])

migr_2015_2019.rename(columns=col_mapping, inplace=True)

### 2009-2014

In [6]:
migr_2009_2014['Bezirk'] = migr_2009_2014['Bezirk'].apply('{:0>2}'.format)
migr_2009_2014.drop('Bezirksname', axis=1, inplace=True)

migr_2009_2014.rename(columns=col_mapping, inplace=True)

### Backfill 2009-2014 NaN values

In [7]:
fill_cols = [c for c in migr_2009_2014.columns if migr_2009_2014[c].isna().any()]

for col in fill_cols:
    for b in migr_2009_2014.Bezirk.unique():
        migr_2009_2014.loc[migr_2009_2014.Bezirk==b, f'{col}_perc'] = migr_2009_2014.loc[migr_2009_2014.Bezirk==b, col] \
                                                                    / migr_2009_2014.loc[migr_2009_2014.Bezirk==b, 'total_foreigners']
        migr_2009_2014.loc[migr_2009_2014.Bezirk==b, f'{col}_perc'] = migr_2009_2014.loc[migr_2009_2014.Bezirk==b, f'{col}_perc'].fillna(method='backfill')
        migr_2009_2014.loc[migr_2009_2014.Bezirk==b, f'{col}_new'] = migr_2009_2014.loc[migr_2009_2014.Bezirk==b, 'total_foreigners'] \
                                                                    * migr_2009_2014.loc[migr_2009_2014.Bezirk==b, f'{col}_perc']
    migr_2009_2014.drop([f'{col}_perc', col], axis=1, inplace=True)
    
rename_columns = {c: c[:-4] for c in migr_2009_2014.columns if '_new' in c}
migr_2009_2014.rename(columns=rename_columns, inplace=True)

### Distributing totals 2009-2014 to PR according to 2015 percentages

In [8]:
cols = ['total_foreigners', 'EU', 'POL', 'ITA', 'former_UDSSR', 'RUS',
        'UKR', 'KAZ', 'former_JUGO', 'TUR', 'Arabic_countries', 'VNM', 'USA',
        'FRA', 'GRC', 'AUT', 'UK', 'BGR', 'BIH', 'HRV', 'SRB', 'Islamic_countries',
        'IRN', 'LBN', 'not_identified', 'ESP', 'ROU', 'SYR']

perc = migr_2015_2019[migr_2015_2019.year==2015].copy()

In [9]:
for col in cols:
    perc[f'{col}_perc'] = perc[col] / perc.groupby('Bezirk')[col].transform(sum)
    perc[f'{col}_perc'].fillna(0, inplace=True)
    perc.drop(col, axis=1, inplace=True)
    
perc.drop('year', axis=1, inplace=True)

In [10]:
migr_2009_2014_new = perc.merge(migr_2009_2014, how='outer', on=['Bezirk'])

In [11]:
for col in cols:
    if col not in migr_2009_2014_new.columns:
        migr_2009_2014_new[col] = 0
    migr_2009_2014_new[col] = migr_2009_2014_new[col] * migr_2009_2014_new[f'{col}_perc']
    migr_2009_2014_new.drop(f'{col}_perc', axis=1, inplace=True)

### Combining all dataframes

In [12]:
migr_final = pd.concat([migr_2009_2014_new, migr_2015_2019], ignore_index=True)

# drop CHN due to incomplete data
migr_final.drop('CHN', axis=1, inplace=True)

### Creating columns for remaining populations

In [13]:
migr_final['EU_other'] = migr_final['EU'] - migr_final[['FRA','GRC', 'ITA', 'AUT', 'ESP', 'UK', 'POL', 'BGR', 'ROU', 'HRV']].sum(axis=1)
migr_final['JUGO_other'] = migr_final['former_JUGO'] - migr_final[['BIH', 'SRB']].sum(axis=1)
migr_final['UDSSR_other'] = migr_final['former_UDSSR'] - migr_final[['RUS', 'UKR', 'KAZ']].sum(axis=1)
migr_final['Islamic_other'] = migr_final['Islamic_countries'] - migr_final[['TUR', 'IRN', 'LBN', 'SYR']].sum(axis=1)
migr_final['other'] = migr_final['total_foreigners'] - migr_final[['EU', 'former_JUGO', 'former_UDSSR', 'Islamic_countries', 'VNM', 'USA', 'not_identified']].sum(axis=1)

In [14]:
import joblib

joblib.dump(migr_final.drop(['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum'], axis=1), './final_migr.csv')

['./final_migr.csv']

In [15]:
migr_final.drop(['Bezirk', 'Prognoseraum', 'Bezirksregion', 'Planungsraum'], axis=1)

,PR,LOR,total_foreigners,EU,POL,ITA,former_UDSSR,RUS,UKR,KAZ,...,LBN,not_identified,ESP,ROU,SYR,EU_other,JUGO_other,UDSSR_other,Islamic_other,other
0,00000000,000000,859253.0,214506.0,87647.0,16962.0,88552.0,39440.0,13310.0,17638.0,...,22731.289777,69299.300466,13793.129747,12872.269383,0.0,2151.872533,19313.047379,18164.0,73075.992114,126597.117213
1,00000000,000000,872132.0,242982.0,92281.0,20425.0,97530.0,43046.0,15096.0,19322.0,...,23072.000000,70338.000000,13999.869459,13065.206687,0.0,20915.923854,24429.000000,20066.0,63333.000000,86405.000000
2,00000000,000000,908188.0,262291.0,95959.0,22320.0,107662.0,44913.0,15676.0,19684.0,...,23698.000000,72594.000000,14578.657180,13605.353238,0.0,27082.989581,27152.000000,27389.0,90932.000000,56539.000000
3,00000000,000000,949183.0,284758.0,99650.0,24932.0,111803.0,46948.0,16349.0,20095.0,...,24355.000000,74811.000000,15236.728033,14219.489800,0.0,34307.782167,28575.000000,28411.0,94681.000000,59788.000000
4,00000000,000000,999348.0,322911.0,104568.0,28234.0,116129.0,49145.0,17033.0,20374.0,...,25104.000000,76808.000000,16042.000000,14971.000000,0.0,54684.000000,30152.000000,29577.0,102269.000000,50619.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4924,12302110,123021,6063.0,1281.0,678.0,158.0,800.0,513.0,37.0,108.0,...,229.000000,416.000000,13.000000,67.000000,233.0,109.000000,150.000000,142.0,1104.000000,107.000000
4925,12302211,123022,948.0,294.0,116.0,22.0,104.0,29.0,12.0,27.0,...,10.000000,110.000000,6.000000,34.000000,13.0,41.000000,39.000000,36.0,96.000000,12.000000
4926,12302212,123022,2947.0,815.0,373.0,65.0,184.0,82.0,32.0,25.0,...,44.000000,132.000000,12.000000,142.000000,498.0,81.000000,104.000000,45.0,479.000000,56.000000
4927,12304313,123043,1796.0,710.0,256.0,38.0,108.0,30.0,30.0,9.0,...,18.000000,89.000000,24.000000,55.000000,105.0,50.000000,131.000000,39.0,259.000000,83.000000
